In [11]:
import torch
import torch.nn as nn
import numpy as np
from collections import deque
import random
import math

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import math
from collections import deque


# ── Replay Buffer ────────────────────────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, next_state, action, reward, done):
        self.buffer.append((state, next_state, action, reward, done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


# ── Model ────────────────────────────────────────────────────────────────────

class DeepQNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1     = nn.Linear(6, 128)
        self.l2     = nn.Linear(128, 64)
        self.output = nn.Linear(64, 4)
        self.relu   = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.l1(x))
        x = self.relu(self.l2(x))
        return self.output(x)


# ── Helpers ──────────────────────────────────────────────────────────────────

def give_state(maze, pos, maze_width=10, maze_length=10):
    """
    Returns a float32 tensor of shape (6,):
      [row/(maze_width-1), col/(maze_length-1), up, down, left, right]
    Maze encoding:
       0  = wall
       1  = free space / S / G
      -1  = out of bounds
    """
    def cell_val(r, c):
        if r < 0 or r >= maze_width or c < 0 or c >= maze_length:
            return -1.0
        v = maze[r][c]
        if v == 'S' or v == 'G':
            return 1.0
        return float(v)  # 0 = wall, 1 = free

    r, c = pos
    up    = cell_val(r - 1, c)
    down  = cell_val(r + 1, c)
    left  = cell_val(r,     c - 1)
    right = cell_val(r,     c + 1)

    return torch.tensor(
        [r / (maze_width - 1), c / (maze_length - 1), up, down, left, right],
        dtype=torch.float32
    ).to(device)


def distance(pos, goal):
    return math.sqrt((goal[0] - pos[0]) ** 2 + (goal[1] - pos[1]) ** 2)


def find_start_goal(maze, maze_width=10, maze_length=10):
    start = goal = None
    for i in range(maze_width):
        for j in range(maze_length):
            if maze[i][j] == 'S':
                start = (i, j)
            if maze[i][j] == 'G':
                goal = (i, j)
        if start and goal:
            break
    return start, goal


def select_action(model, state, epsilon):
    if random.random() < epsilon:
        return random.randint(0, 3)
    with torch.no_grad():
        return model(state).argmax().item()


def apply_action(pos, action, maze, maze_width=10, maze_length=10):
    """
    Returns new position after applying action.
    action: 0=up, 1=down, 2=left, 3=right
    Stays in place if target cell is a wall or out of bounds.
    """
    r, c = pos
    deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dr, dc = deltas[action]
    nr, nc = r + dr, c + dc

    if 0 <= nr < maze_width and 0 <= nc < maze_length and maze[nr][nc] != 0:
        return (nr, nc)
    return pos   # hit wall or out of bounds → stay


# ── Learn (one training step) ────────────────────────────────────────────────

def learn(model, buffer, optimizer, criterion, batch_size=64, gamma=0.99):
    batch = buffer.sample(batch_size)

    states      = torch.stack([b[0] for b in batch]).to(device)          # (B, 6)
    next_states = torch.stack([b[1] for b in batch]).to(device)          # (B, 6)
    actions     = torch.tensor([b[2] for b in batch], dtype=torch.long).to(device)    # (B,)
    rewards     = torch.tensor([b[3] for b in batch], dtype=torch.float32).to(device) # (B,)
    dones       = torch.tensor([b[4] for b in batch], dtype=torch.float32).to(device) # (B,)

    # Prediction: Q(s, a_taken)
    q_values    = model(states)                                # (B, 4)
    predictions = q_values[range(len(actions)), actions]      # (B,)

    # Target: r + gamma * max Q(s') * (1 - done)
    with torch.no_grad():
        next_q   = model(next_states).max(dim=1).values       # (B,)
    targets = rewards + gamma * next_q * (1 - dones)          # (B,)

    loss = criterion(predictions, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


# ── Train ────────────────────────────────────────────────────────────────────

def train(
    mazes,
    model,
    gamma             = 0.99,
    batch_size        = 64,
    max_episodes      = 2000,
    capacity          = 10000,
    min_buffer_size   = 500,
    epsilon_start     = 0.99,
    epsilon_min       = 0.01,
    epsilon_decay     = 0.9997,
    max_steps         = 200,
    maze_length       = 10,
    maze_width        = 10,
    n_episodic_reward = 100,
):
    epsilon   = epsilon_start
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    buffer    = ReplayBuffer(capacity=capacity)

    all_reward_history = []

    for maze_idx, maze in enumerate(mazes):
        start, goal = find_start_goal(maze, maze_width, maze_length)
        assert start and goal, f"Maze {maze_idx+1} missing 'S' or 'G'."

        reached            = 0
        total_reward_history = []
        epsilon = max(epsilon, 0.899)
        buffer = ReplayBuffer(capacity = capacity)

        for episode in range(max_episodes):
            curr_pos     = start
            visited      = {curr_pos}
            done         = 0
            total_reward = 0.0

            for step in range(max_steps):
                prev_pos    = curr_pos
                state       = give_state(maze, curr_pos, maze_width, maze_length)
                action      = select_action(model, state, epsilon)
                curr_pos    = apply_action(curr_pos, action, maze, maze_width, maze_length)
                next_state  = give_state(maze, curr_pos, maze_width, maze_length)

                # ── Reward ───────────────────────────────────────────────────
                if curr_pos == prev_pos:          # hit wall (stayed in place)
                    reward = -0.5
                elif curr_pos == goal:
                    reward  = 1.0
                    done    = 1
                    reached += 1
                elif curr_pos in visited:         # revisiting a cell
                    reward = -0.3
                else:
                    reward = -0.05

                # Shaping: bonus for getting closer to goal
                if distance(curr_pos, goal) < distance(prev_pos, goal):
                    reward += 0.1

                visited.add(curr_pos)
                total_reward += reward

                # ── Buffer & learn ───────────────────────────────────────────
                buffer.push(state, next_state, action, reward, done)
                if len(buffer) >= min_buffer_size:
                    learn(model, buffer, optimizer, criterion, batch_size, gamma)

                if done:
                    break

            total_reward_history.append(total_reward)

            if episode % n_episodic_reward == 0:
                avg = sum(total_reward_history[-n_episodic_reward:]) / min(len(total_reward_history), n_episodic_reward)
                print(
                    f"Maze {maze_idx+1} | Episode {episode:4d} | "
                    f"Epsilon {epsilon:.3f} | "
                    f"Reward {total_reward:7.2f} | "
                    f"Avg(last {n_episodic_reward}) {avg:7.2f}"
                )

            epsilon = max(epsilon_min, epsilon * epsilon_decay)

        print(f"\nMaze {maze_idx+1} finished — reached goal {reached}/{max_episodes} times.\n")
        all_reward_history.append(total_reward_history)

    return all_reward_history


# ── Evaluate ─────────────────────────────────────────────────────────────────

def evaluate(model, maze, max_steps=200, maze_length=10, maze_width=10):
    start, goal = find_start_goal(maze, maze_width, maze_length)
    assert start and goal, "Maze missing 'S' or 'G'."

    curr_pos = start
    path     = [curr_pos]

    for step in range(max_steps):
        if curr_pos == goal:
            break
        state    = give_state(maze, curr_pos, maze_width, maze_length)
        with torch.no_grad():
            action = model(state).argmax().item()   # greedy, no epsilon
        curr_pos = apply_action(curr_pos, action, maze, maze_width, maze_length)
        path.append(curr_pos)

    reached = (curr_pos == goal)
    print(f"{'Goal reached' if reached else 'Goal NOT reached'} in {len(path)-1} steps.")
    return path, reached

In [14]:
model = DeepQNetwork().to(device)

In [15]:
train_mazes = [
    # Maze 1 — simplest, open corridors, easy path
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 2 — same skeleton, one new wall added mid-path
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 0, 1, 0, 1, 0],  # row 6 col 4: 1→0 (open shortcut)
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 0],  # row 7: new wall forces detour
        [0, 1, 1, 1, 0, 0, 0, 0, 'G', 0],  # bottom path changed
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 3 — top corridor blocked, must use bottom route
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 0, 1, 1, 1, 1, 0],  # col 4 blocked
        [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1, 0],
        [0, 0, 0, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 1, 0, 1, 1, 1, 0, 1, 0],
        [0, 1, 1, 1, 0, 0, 0, 0, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 4 — slight reroute, introduces a dead end trap
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 0, 1, 1, 1, 1, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 1, 0, 0, 1, 0],  # dead end at col 5
        [0, 1, 1, 1, 0, 1, 0, 1, 1, 0],
        [0, 0, 0, 1, 0, 1, 0, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 1, 0, 1, 0],
        [0, 1, 1, 1, 0, 0, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 5 — two dead ends, path requires backtracking awareness
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 0, 1, 1, 0, 1, 0],
        [0, 0, 0, 1, 0, 1, 0, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 1, 1, 1, 0],
        [0, 1, 0, 1, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1, 0],
        [0, 0, 0, 0, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 6 — longer mandatory path, fewer shortcuts
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 0, 1, 1, 1, 0, 1, 0],
        [0, 0, 1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 1, 1, 0],
        [0, 1, 1, 1, 1, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1, 1, 1, 0, 1, 0],
        [0, 1, 1, 0, 0, 0, 1, 0, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 7 — spiral-like structure
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 1, 0, 0, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 8 — spiral with one extra wall forcing longer route
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1, 0],  # extra wall row 7 col 1-4
        [0, 0, 0, 0, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 9 — denser walls, narrower corridors
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 0, 1, 0, 1, 1, 1, 0],
        [0, 0, 1, 0, 1, 0, 0, 0, 1, 0],
        [0, 1, 1, 0, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1, 0],
        [0, 0, 0, 0, 1, 0, 1, 1, 1, 0],
        [0, 1, 1, 0, 1, 0, 0, 0, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Maze 10 — most complex, multiple dead ends, longest path
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 0, 1, 0, 1, 0, 1, 0],
        [0, 0, 1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 1, 1, 0],
        [0, 1, 1, 1, 1, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1, 1, 1, 0, 1, 0],
        [0, 1, 1, 0, 0, 0, 1, 0, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
]

test_mazes = [
    # Test 1 — similar to maze 1-2, slightly different middle section
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 0],
        [0, 1, 1, 0, 0, 0, 0, 0, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Test 2 — similar to maze 3-4, different dead end placement
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 0, 1, 1, 1, 1, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 1, 1, 1, 0, 0, 1, 0],
        [0, 1, 0, 0, 0, 1, 1, 0, 1, 0],
        [0, 1, 1, 1, 0, 0, 1, 0, 1, 0],
        [0, 0, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Test 3 — similar to maze 5-6, new shortcut available
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 0, 1, 1, 1, 0, 1, 0],
        [0, 0, 1, 0, 0, 0, 1, 0, 1, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 1, 1, 0],
        [0, 1, 1, 0, 1, 0, 0, 0, 1, 0],
        [0, 0, 1, 0, 1, 1, 1, 0, 1, 0],
        [0, 1, 1, 0, 0, 0, 1, 0, 1, 0],
        [0, 0, 0, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Test 4 — similar to maze 7-8, spiral variant
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 1, 1, 0, 0, 1, 0],
        [0, 1, 0, 0, 0, 1, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 1, 0, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    # Test 5 — similar to maze 9-10, dense walls novel layout
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 'S', 1, 0, 1, 1, 0, 1, 1, 0],
        [0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
        [0, 1, 1, 1, 0, 1, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 0, 1, 1, 1, 0],
        [0, 1, 1, 0, 1, 0, 0, 0, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 'G', 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
]

In [16]:
EPISODES        = 2000
MAX_STEPS       = 200
BATCH_SIZE      = 64
BUFFER_CAPACITY = 10000
MIN_BUFFER_SIZE = 1000
LR              = 0.001
GAMMA           = 0.99
EPSILON         = 0.99
EPSILON_DECAY   = 0.9997
EPSILON_MIN     = 0.01

In [17]:
train(
    train_mazes,
    model,
    gamma = GAMMA,
    batch_size = BATCH_SIZE,
    max_episodes = EPISODES,
    capacity = BUFFER_CAPACITY,
    min_buffer_size = MIN_BUFFER_SIZE,
    epsilon_start = EPSILON,
    epsilon_min = EPSILON_MIN,
    epsilon_decay = EPSILON_DECAY,
    max_steps = MAX_STEPS,
    maze_length = 10,
    maze_width = 10,
    n_episodic_reward = 100
)

Maze 1 | Episode    0 | Epsilon 0.990 | Reward  -77.95 | Avg(last 100)  -77.95
Maze 1 | Episode  100 | Epsilon 0.961 | Reward  -52.75 | Avg(last 100)  -60.38
Maze 1 | Episode  200 | Epsilon 0.932 | Reward  -53.75 | Avg(last 100)  -48.76
Maze 1 | Episode  300 | Epsilon 0.905 | Reward  -57.75 | Avg(last 100)  -39.32
Maze 1 | Episode  400 | Epsilon 0.878 | Reward  -36.25 | Avg(last 100)  -30.06
Maze 1 | Episode  500 | Epsilon 0.852 | Reward  -22.75 | Avg(last 100)  -25.56
Maze 1 | Episode  600 | Epsilon 0.827 | Reward  -27.75 | Avg(last 100)  -22.86
Maze 1 | Episode  700 | Epsilon 0.802 | Reward   -6.75 | Avg(last 100)  -21.01
Maze 1 | Episode  800 | Epsilon 0.779 | Reward  -28.25 | Avg(last 100)  -17.05
Maze 1 | Episode  900 | Epsilon 0.756 | Reward  -18.75 | Avg(last 100)  -13.59
Maze 1 | Episode 1000 | Epsilon 0.733 | Reward   -9.25 | Avg(last 100)  -13.69
Maze 1 | Episode 1100 | Epsilon 0.712 | Reward   -5.75 | Avg(last 100)  -10.63
Maze 1 | Episode 1200 | Epsilon 0.691 | Reward  -12.

[[-77.95,
  -75.85000000000001,
  -75.75000000000001,
  -76.85000000000001,
  -41.25000000000002,
  -74.35000000000001,
  -56.75,
  -73.54999999999998,
  -73.99999999999999,
  -65.75000000000003,
  -70.85000000000001,
  -55.75000000000002,
  -72.80000000000003,
  -74.0,
  -76.75000000000001,
  -56.75000000000002,
  -54.25000000000002,
  -76.99999999999999,
  -57.25000000000002,
  -76.40000000000002,
  -72.25,
  -57.25000000000001,
  -75.39999999999998,
  -47.75000000000003,
  -66.85000000000001,
  -71.65,
  -37.75000000000003,
  -37.25000000000003,
  -56.25000000000001,
  -71.65000000000002,
  -73.80000000000001,
  -21.249999999999993,
  -72.34999999999998,
  -75.9,
  -72.8,
  -28.24999999999999,
  -70.3,
  -71.10000000000001,
  -44.75,
  -75.80000000000003,
  -71.25000000000001,
  -30.249999999999993,
  -32.24999999999999,
  -68.45,
  -60.75000000000002,
  -71.45000000000002,
  -15.249999999999991,
  -74.64999999999999,
  -73.25000000000003,
  -78.85000000000001,
  -74.99999999999999,

In [22]:
evaluate(model, test_mazes[4])

Goal reached in 14 steps.


([(1, 1),
  (1, 2),
  (2, 2),
  (3, 2),
  (3, 3),
  (4, 3),
  (5, 3),
  (5, 4),
  (6, 4),
  (7, 4),
  (8, 4),
  (8, 5),
  (8, 6),
  (8, 7),
  (8, 8)],
 True)